# scModuFuse: Single-Cell Modular Graph Fusion for Spatial Multi-Omics

**scModuFuse** is a flexible single-cell spatial multi-omics integration framework that decouples **Within-Modality** and **Between-Modality** fusion.
It supports dynamic selection of state-of-the-art fusion mechanisms:
- **Hierarchical Fusion** (Multi-stage non-linear projection derived from ARISE)
- **QKV Cross-Attention Fusion** (Query-Key-Value attention mechanism)
- **Gated Fusion** (Sigmoidal elementwise gating)
- **Softmax Attention Fusion** (Classic attention weight matrix)

---


In [ ]:
%pip install scanpy anndata scikit-misc rpy2 gdown


In [ ]:
import os
import scipy
import anndata
import sklearn
import torch
import random
import copy
import numpy as np
import scanpy as sc
import pandas as pd
from typing import Optional
import scipy.sparse as sp
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module
from torch.backends import cudnn
from scipy.sparse import coo_matrix
from scipy.sparse import issparse
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    silhouette_score
)
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

def init_weights(*params):
    for param in params:
        torch.nn.init.xavier_uniform_(param)

def fix_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

def construct_neighbor_graph(adata_omics1, adata_omics2, datatype='SPOTS', n_neighbors=3):
    if datatype in ['Stereo-CITE-seq', 'Spatial-epigenome-transcriptome']:
        n_neighbors = 6
    cell_position_omics1 = adata_omics1.obsm['spatial']
    adata_omics1.uns['adj_spatial'] = construct_graph_by_coordinate(cell_position_omics1, n_neighbors=n_neighbors)
    cell_position_omics2 = adata_omics2.obsm['spatial']
    adata_omics2.uns['adj_spatial'] = construct_graph_by_coordinate(cell_position_omics2, n_neighbors=n_neighbors)
    feature_graph_omics1, feature_graph_omics2 = construct_graph_by_feature(adata_omics1, adata_omics2)
    adata_omics1.obsm['adj_feature'], adata_omics2.obsm['adj_feature'] = feature_graph_omics1, feature_graph_omics2
    data = {'adata_omics1': adata_omics1, 'adata_omics2': adata_omics2}
    return data

def pca(adata, use_reps=None, n_comps=10):
    from sklearn.decomposition import PCA
    from scipy.sparse import csc_matrix, csr_matrix
    pca_model = PCA(n_components=n_comps)
    if use_reps is not None:
        feat_pca = pca_model.fit_transform(adata.obsm[use_reps])
    else:
        if isinstance(adata.X, csc_matrix) or isinstance(adata.X, csr_matrix):
            feat_pca = pca_model.fit_transform(adata.X.toarray())
        else:
            feat_pca = pca_model.fit_transform(adata.X)
    return feat_pca

def clr_normalize_each_cell(adata, inplace=True):
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x)) if len(x) > 0 else 1.0
        return np.log1p(x / exp)
    if not inplace:
        adata = adata.copy()
    adata.X = np.apply_along_axis(
        seurat_clr, 1, (adata.X.toarray() if scipy.sparse.issparse(adata.X) else np.array(adata.X))
    )
    return adata

def construct_graph_by_feature(adata_omics1, adata_omics2, k=20, mode="connectivity", metric="correlation", include_self=False):
    feature_graph_omics1 = kneighbors_graph(adata_omics1.obsm['feat'], k, mode=mode, metric=metric, include_self=include_self)
    feature_graph_omics2 = kneighbors_graph(adata_omics2.obsm['feat'], k, mode=mode, metric=metric, include_self=include_self)
    return feature_graph_omics1, feature_graph_omics2

def construct_graph_by_coordinate(cell_position, n_neighbors=3):
    nbrs = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(cell_position)
    _, indices = nbrs.kneighbors(cell_position)
    x = indices[:, 0].repeat(n_neighbors)
    y = indices[:, 1:].flatten()
    adj = pd.DataFrame({'x': x, 'y': y, 'value': np.ones(x.size)})
    return adj

def transform_adjacent_matrix(adjacent):
    n_spot = adjacent['x'].max() + 1
    adj = coo_matrix((adjacent['value'], (adjacent['x'], adjacent['y'])), shape=(n_spot, n_spot))
    return adj

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse.FloatTensor(indices, values, shape)

def preprocess_graph(adj):
    adj = sp.coo_matrix(adj)
    adj_ = adj + sp.eye(adj.shape[0])
    rowsum = np.array(adj_.sum(1))
    degree_mat_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    adj_normalized = adj_.dot(degree_mat_inv_sqrt).transpose().dot(degree_mat_inv_sqrt).tocoo()
    return sparse_mx_to_torch_sparse_tensor(adj_normalized)

def adjacent_matrix_preprocessing(adata_omics1, adata_omics2):
    adj_spatial_omics1 = transform_adjacent_matrix(adata_omics1.uns['adj_spatial']).toarray()
    adj_spatial_omics2 = transform_adjacent_matrix(adata_omics2.uns['adj_spatial']).toarray()
    adj_spatial_omics1 = np.where((adj_spatial_omics1 + adj_spatial_omics1.T) > 1, 1, adj_spatial_omics1 + adj_spatial_omics1.T)
    adj_spatial_omics2 = np.where((adj_spatial_omics2 + adj_spatial_omics2.T) > 1, 1, adj_spatial_omics2 + adj_spatial_omics2.T)
    adj_spatial_omics1 = preprocess_graph(adj_spatial_omics1)
    adj_spatial_omics2 = preprocess_graph(adj_spatial_omics2)
    adj_feature_omics1 = torch.FloatTensor(adata_omics1.obsm['adj_feature'].copy().toarray())
    adj_feature_omics2 = torch.FloatTensor(adata_omics2.obsm['adj_feature'].copy().toarray())
    adj_feature_omics1 = preprocess_graph(np.where((adj_feature_omics1 + adj_feature_omics1.T) > 1, 1, adj_feature_omics1 + adj_feature_omics1.T))
    adj_feature_omics2 = preprocess_graph(np.where((adj_feature_omics2 + adj_feature_omics2.T) > 1, 1, adj_feature_omics2 + adj_feature_omics2.T))
    return {'adj_spatial_omics1': adj_spatial_omics1, 'adj_spatial_omics2': adj_spatial_omics2,
            'adj_feature_omics1': adj_feature_omics1, 'adj_feature_omics2': adj_feature_omics2}

def lsi(adata: anndata.AnnData, n_components: int = 20, use_highly_variable: Optional[bool] = None, **kwargs):
    if use_highly_variable is None:
        use_highly_variable = "highly_variable" in adata.var
    adata_use = adata[:, adata.var["highly_variable"]] if use_highly_variable else adata
    X = tfidf(adata_use.X)
    X_norm = sklearn.preprocessing.Normalizer(norm="l1").fit_transform(X)
    X_norm = np.log1p(X_norm * 1e4)
    X_lsi = sklearn.utils.extmath.randomized_svd(X_norm, n_components, **kwargs)[0]
    X_lsi -= X_lsi.mean(axis=1, keepdims=True)
    X_lsi /= X_lsi.std(axis=1, ddof=1, keepdims=True)
    adata.obsm["X_lsi"] = X_lsi[:, 1:]

def tfidf(X):
    idf = X.shape[0] / X.sum(axis=0)
    if scipy.sparse.issparse(X):
        tf = X.multiply(1 / X.sum(axis=1))
        return tf.multiply(idf)
    else:
        tf = X / X.sum(axis=1, keepdims=True)
        return tf * idf

# ---- scModuFuse Fusion Modules ----

class AttentionLayer(Module):
    """Standard Softmax Attention Layer."""
    def __init__(self, in_feat, out_feat, dropout=0.0, act=F.relu):
        super(AttentionLayer, self).__init__()
        self.in_feat = in_feat
        self.out_feat = out_feat
        self.w_omega = Parameter(torch.FloatTensor(in_feat, out_feat))
        self.u_omega = Parameter(torch.FloatTensor(out_feat, 1))
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.w_omega)
        torch.nn.init.xavier_uniform_(self.u_omega)

    def forward(self, emb1, emb2):
        emb = []
        emb.append(torch.unsqueeze(torch.squeeze(emb1), dim=1))
        emb.append(torch.unsqueeze(torch.squeeze(emb2), dim=1))
        self.emb = torch.cat(emb, dim=1)
        self.v = torch.tanh(torch.matmul(self.emb, self.w_omega))
        self.vu = torch.matmul(self.v, self.u_omega)
        self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6, dim=-1)
        emb_combined = torch.matmul(torch.transpose(self.emb, 1, 2), torch.unsqueeze(self.alpha, -1))
        return torch.squeeze(emb_combined), self.alpha

class GatedFusionLayer(nn.Module):
    """Gated Modality Fusion Layer."""
    def __init__(self, dim):
        super(GatedFusionLayer, self).__init__()
        self.fc_s = nn.Linear(dim, dim, bias=True)
        self.fc_f = nn.Linear(dim, dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fc_s.weight)
        nn.init.xavier_uniform_(self.fc_f.weight)
        nn.init.zeros_(self.fc_s.bias)
        nn.init.zeros_(self.fc_f.bias)

    def forward(self, emb_s, emb_f):
        gate = torch.sigmoid(self.fc_s(emb_s) + self.fc_f(emb_f))
        fused = gate * emb_s + (1.0 - gate) * emb_f
        g = gate.mean(dim=-1, keepdim=True)
        alpha = torch.cat([g, 1.0 - g], dim=-1)
        return fused, alpha

class QKVCrossFusionLayer(nn.Module):
    """QKV Cross-Modality Attention Fusion Layer."""
    def __init__(self, dim, attention_type='local'):
        super().__init__()
        self.dim = dim
        self.attention_type = attention_type
        self.scale = dim ** -0.5
        self.q_proj1 = nn.Linear(dim, dim, bias=False)
        self.k_proj1 = nn.Linear(dim, dim, bias=False)
        self.v_proj1 = nn.Linear(dim, dim, bias=False)
        self.q_proj2 = nn.Linear(dim, dim, bias=False)
        self.k_proj2 = nn.Linear(dim, dim, bias=False)
        self.v_proj2 = nn.Linear(dim, dim, bias=False)
        self.fc_out = nn.Linear(2 * dim, dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.q_proj1.weight)
        nn.init.xavier_uniform_(self.k_proj1.weight)
        nn.init.xavier_uniform_(self.v_proj1.weight)
        nn.init.xavier_uniform_(self.q_proj2.weight)
        nn.init.xavier_uniform_(self.k_proj2.weight)
        nn.init.xavier_uniform_(self.v_proj2.weight)
        nn.init.xavier_uniform_(self.fc_out.weight)
        nn.init.zeros_(self.fc_out.bias)

    def forward(self, emb1, emb2):
        q1, k1, v1 = self.q_proj1(emb1), self.k_proj1(emb1), self.v_proj1(emb1)
        q2, k2, v2 = self.q_proj2(emb2), self.k_proj2(emb2), self.v_proj2(emb2)
        if self.attention_type == 'global':
            attn_scores1 = torch.matmul(q2, k1.T) * self.scale
            z1 = torch.matmul(F.softmax(attn_scores1, dim=-1), v1)
            attn_scores2 = torch.matmul(q1, k2.T) * self.scale
            z2 = torch.matmul(F.softmax(attn_scores2, dim=-1), v2)
            alpha = torch.cat([F.softmax(attn_scores1, dim=-1).mean(dim=-1, keepdim=True), F.softmax(attn_scores2, dim=-1).mean(dim=-1, keepdim=True)], dim=-1)
        else:
            w1 = torch.sigmoid((q2 * k1).sum(dim=-1, keepdim=True) * self.scale)
            w2 = torch.sigmoid((q1 * k2).sum(dim=-1, keepdim=True) * self.scale)
            z1, z2 = w1 * v1, w2 * v2
            alpha = torch.cat([w1, w2], dim=-1)
        return self.fc_out(torch.cat([z1, z2], dim=-1)), alpha

class HierarchicalFusionLayer(nn.Module):
    """
    Hierarchical Fusion Layer based on the ARISE architecture.
    Performs multi-stage hierarchical feature fusion using non-linear projection
    and dynamic gating weight estimation across within-modality and between-modality representations.
    """
    def __init__(self, dim, hidden_dim=None, act_fn=F.relu):
        super(HierarchicalFusionLayer, self).__init__()
        if hidden_dim is None:
            hidden_dim = dim
        self.dim = dim
        self.act_fn = act_fn
        
        self.fusion_fc1 = nn.Linear(2 * dim, hidden_dim, bias=True)
        self.fusion_fc2 = nn.Linear(hidden_dim, dim, bias=True)
        self.attn_fc = nn.Linear(dim, 2, bias=True)
        
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fusion_fc1.weight)
        nn.init.zeros_(self.fusion_fc1.bias)
        nn.init.xavier_uniform_(self.fusion_fc2.weight)
        nn.init.zeros_(self.fusion_fc2.bias)
        nn.init.xavier_uniform_(self.attn_fc.weight)
        nn.init.zeros_(self.attn_fc.bias)

    def forward(self, emb1, emb2):
        combined = torch.cat([emb1, emb2], dim=-1)
        h = self.act_fn(self.fusion_fc1(combined))
        fused = self.fusion_fc2(h)
        alpha = F.softmax(self.attn_fc(fused), dim=-1)
        return fused, alpha

def get_fusion_layer(fusion_type: str, dim: int, attention_type: str = 'local'):
    """
    Factory function to instantiate fusion layer modules dynamically.
    Options for fusion_type: 'gated', 'qkv', 'hierarchical', 'attention'
    """
    fusion_type = fusion_type.lower()
    if fusion_type == 'hierarchical':
        return HierarchicalFusionLayer(dim)
    elif fusion_type == 'qkv':
        return QKVCrossFusionLayer(dim, attention_type=attention_type)
    elif fusion_type == 'gated':
        return GatedFusionLayer(dim)
    elif fusion_type == 'attention':
        return AttentionLayer(dim, dim)
    else:
        raise ValueError(f"Unknown fusion type: '{fusion_type}'. Valid options are ['gated', 'qkv', 'hierarchical', 'attention']")

class Encoder(Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.weight = Parameter(torch.FloatTensor(in_feat, out_feat))
        torch.nn.init.xavier_uniform_(self.weight)
    def forward(self, feat, adj):
        return torch.spmm(adj, torch.mm(feat, self.weight))

class Decoder(Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.weight = Parameter(torch.FloatTensor(in_feat, out_feat))
        torch.nn.init.xavier_uniform_(self.weight)
    def forward(self, feat, adj):
        return torch.spmm(adj, torch.mm(feat, self.weight))

class Encoder_overall(Module):
    """
    scModuFuse Multimodal Graph Encoder supporting independent Within-Modality and Between-Modality fusion techniques.
    """
    def __init__(self, dim_in_feat_omics1, dim_out_feat_omics1, dim_in_feat_omics2, dim_out_feat_omics2, 
                 within_fusion='gated', between_fusion='qkv', attention_type='local'):
        super().__init__()
        self.encoder_omics1 = Encoder(dim_in_feat_omics1, dim_out_feat_omics1)
        self.decoder_omics1 = Decoder(dim_out_feat_omics1, dim_in_feat_omics1)
        self.encoder_omics2 = Encoder(dim_in_feat_omics2, dim_out_feat_omics2)
        self.decoder_omics2 = Decoder(dim_out_feat_omics2, dim_in_feat_omics2)

        # Within-Modality Fusion Layers (Omics 1 and Omics 2)
        self.atten_omics1 = get_fusion_layer(within_fusion, dim_out_feat_omics1, attention_type=attention_type)
        self.atten_omics2 = get_fusion_layer(within_fusion, dim_out_feat_omics2, attention_type=attention_type)
        
        # Between-Modality Cross Fusion Layer
        self.atten_cross = get_fusion_layer(between_fusion, dim_out_feat_omics1, attention_type=attention_type)

    def forward(self, f1, f2, adj_s1, adj_f1, adj_s2, adj_f2):
        emb_s1, emb_s2 = self.encoder_omics1(f1, adj_s1), self.encoder_omics2(f2, adj_s2)
        emb_f1, emb_f2 = self.encoder_omics1(f1, adj_f1), self.encoder_omics2(f2, adj_f2)
        
        # Within-modality fusion
        emb_o1, alpha_o1 = self.atten_omics1(emb_s1, emb_f1)
        emb_o2, alpha_o2 = self.atten_omics2(emb_s2, emb_f2)
        
        # Between-modality cross fusion
        emb_comb, alpha_cross = self.atten_cross(emb_o1, emb_o2)
        
        recon1, recon2 = self.decoder_omics1(emb_comb, adj_s1), self.decoder_omics2(emb_comb, adj_s2)
        cross1 = self.encoder_omics2(self.decoder_omics2(emb_o1, adj_s2), adj_s2)
        cross2 = self.encoder_omics1(self.decoder_omics1(emb_o2, adj_s1), adj_s1)
        return {'emb_latent_omics1': emb_o1, 'emb_latent_omics2': emb_o2, 'emb_latent_combined': emb_comb,
                'emb_recon_omics1': recon1, 'emb_recon_omics2': recon2, 'emb_latent_omics1_across_recon': cross1,
                'emb_latent_omics2_across_recon': cross2, 'alpha_omics1': alpha_o1, 'alpha_omics2': alpha_o2, 'alpha': alpha_cross}

class Train_scModuFuse:
    """
    Master Training Engine for scModuFuse.
    """
    def __init__(self, data, datatype='SPOTS', device=torch.device('cpu'), epochval=None, random_seed=2022, 
                 learning_rate=0.0001, weight_decay=0.0, epochs=600, dim_output=64, weight_factors=[1, 5, 1, 1], 
                 within_fusion='gated', between_fusion='qkv', attention_type='local'):
        self.data = data.copy()
        self.datatype = datatype
        self.device = device
        self.random_seed = random_seed
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.dim_output = dim_output
        self.weight_factors = weight_factors
        self.within_fusion = within_fusion
        self.between_fusion = between_fusion
        self.attention_type = attention_type
        self.loss_history = []
        self.adata_omics1, self.adata_omics2 = self.data['adata_omics1'], self.data['adata_omics2']
        self.adj = adjacent_matrix_preprocessing(self.adata_omics1, self.adata_omics2)
        self.adj_spatial_omics1 = self.adj['adj_spatial_omics1'].to(self.device)
        self.adj_spatial_omics2 = self.adj['adj_spatial_omics2'].to(self.device)
        self.adj_feature_omics1 = self.adj['adj_feature_omics1'].to(self.device)
        self.adj_feature_omics2 = self.adj['adj_feature_omics2'].to(self.device)
        self.features_omics1 = torch.FloatTensor(self.adata_omics1.obsm['feat'].copy()).to(self.device)
        self.features_omics2 = torch.FloatTensor(self.adata_omics2.obsm['feat'].copy()).to(self.device)
        self.dim_input1, self.dim_input2 = self.features_omics1.shape[1], self.features_omics2.shape[1]
        if self.datatype == 'SPOTS': self.epochs, self.weight_factors = 600, [1, 5, 1, 1]
        elif self.datatype == '10x': self.epochs, self.weight_factors = 200, [1, 5, 1, 10]
        elif self.datatype == 'Spatial-epigenome-transcriptome': self.epochs, self.weight_factors = 1600, [1, 5, 1, 1]
        if epochval is not None: self.epochs = epochval

    def train(self):
        self.model = Encoder_overall(
            self.dim_input1, self.dim_output, self.dim_input2, self.dim_output, 
            within_fusion=self.within_fusion, between_fusion=self.between_fusion, attention_type=self.attention_type
        ).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), self.learning_rate, weight_decay=self.weight_decay)
        for epoch in tqdm(range(self.epochs)):
            self.model.train()
            results = self.model(self.features_omics1, self.features_omics2, self.adj_spatial_omics1, self.adj_feature_omics1, self.adj_spatial_omics2, self.adj_feature_omics2)
            loss1 = F.mse_loss(self.features_omics1, results['emb_recon_omics1'])
            loss2 = F.mse_loss(self.features_omics2, results['emb_recon_omics2'])
            loss3 = F.mse_loss(results['emb_latent_omics1'], results['emb_latent_omics1_across_recon'])
            loss4 = F.mse_loss(results['emb_latent_omics2'], results['emb_latent_omics2_across_recon'])
            loss = self.weight_factors[0]*loss1 + self.weight_factors[1]*loss2 + self.weight_factors[2]*loss3 + self.weight_factors[3]*loss4
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.loss_history.append(loss.item())
        with torch.no_grad():
            self.model.eval()
            results = self.model(self.features_omics1, self.features_omics2, self.adj_spatial_omics1, self.adj_feature_omics1, self.adj_spatial_omics2, self.adj_feature_omics2)
        return {'emb_latent_omics1': F.normalize(results['emb_latent_omics1'], p=2, eps=1e-12, dim=1).detach().cpu().numpy(),
                'emb_latent_omics2': F.normalize(results['emb_latent_omics2'], p=2, eps=1e-12, dim=1).detach().cpu().numpy(),
                'scModuFuse': F.normalize(results['emb_latent_combined'], p=2, eps=1e-12, dim=1).detach().cpu().numpy(),
                'alpha_omics1': results['alpha_omics1'].detach().cpu().numpy(),
                'alpha_omics2': results['alpha_omics2'].detach().cpu().numpy(),
                'alpha': results['alpha'].detach().cpu().numpy(), 'loss_history': self.loss_history}

def mclust_R(adata, num_cluster, modelNames='EEE', used_obsm='emb_pca', random_seed=2020):
    import rpy2.robjects as robjects
    from rpy2.robjects import pandas2ri, default_converter
    from rpy2.robjects.conversion import localconverter
    np.random.seed(random_seed)
    robjects.r.library("mclust")
    robjects.r["set.seed"](random_seed)
    rmclust = robjects.r["Mclust"]
    X = np.array(adata.obsm[used_obsm], dtype=np.float64)
    df = pd.DataFrame(X, columns=[f'PC{i+1}' for i in range(X.shape[1])])
    subset_size = min(300, X.shape[0])
    subset_indices = robjects.IntVector(list(np.random.choice(range(1, X.shape[0] + 1), subset_size, replace=False)))
    init_list = robjects.ListVector({'subset': subset_indices})
    with localconverter(default_converter + pandas2ri.converter):
        res = rmclust(df, G=num_cluster, modelNames=modelNames, initialization=init_list)
    mclust_res = np.array(res['classification'])
    adata.obs['mclust'] = mclust_res
    adata.obs['mclust'] = adata.obs['mclust'].astype('int').astype('str').astype('category')
    return adata

def clustering(adata, n_clusters=7, key='emb', add_key='scModuFuse', method='mclust', start=0.1, end=3.0, increment=0.01, use_pca=False, n_comps=20, random_seed=2020):
    if use_pca: adata.obsm[key + '_pca'] = pca(adata, use_reps=key, n_comps=n_comps)
    if method == 'mclust':
        used = key + '_pca' if use_pca else key
        adata = mclust_R(adata, used_obsm=used, num_cluster=n_clusters, random_seed=random_seed)
        adata.obs[add_key] = adata.obs['mclust']


# Benchmark Harness across 6 Spatial Multi-Omics Datasets and 10 Random Seeds


In [ ]:
# Setup R_HOME and rpy2
os.environ['R_HOME'] = '/usr/lib/R'
os.environ['PATH'] = '/usr/lib/R/bin:' + os.environ['PATH']
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, default_converter
import rpy2.robjects.conversion as cv
cv.set_conversion(default_converter + pandas2ri.converter)
robjects.r.options(warn=-1)
robjects.r('''
if (!requireNamespace("mclust", quietly = TRUE)) {
    install.packages("mclust", repos="https://cloud.r-project.org")
}
library(mclust)
''')

# ---- Configuration for Independent Within and Between Fusion ----
# Options for WITHIN_FUSION & BETWEEN_FUSION: 'gated', 'qkv', 'hierarchical', 'attention'
WITHIN_FUSION = 'gated'
BETWEEN_FUSION = 'qkv'

choices = [
    ("10x_human_lymph_node_A1", "https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC"),
    ("10x_human_lymph_node_D1", "https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz"),
    ("Mouse_Brain_E11_S1", "https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo"),
    ("Mouse_Brain_E13_S1", "https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud"),
    ("Mouse_Brain_E15_S1", "https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD"),
    ("Mouse_Brain_E18_S1", "https://drive.google.com/drive/folders/1Xj1LNIAY93biS6JIMKNRODn5GvtCKADB")
]

DATASET_INDICES = [0, 1, 2, 3, 4, 5]
SEEDS = [42, 0, 1, 7, 123, 1234, 2022, 2023, 2024, 1337]
tool = 'mclust'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

os.makedirs("results", exist_ok=True)
all_results = []

print(f"Executing scModuFuse with Within-Fusion: '{WITHIN_FUSION}' | Between-Fusion: '{BETWEEN_FUSION}'")

for dataset_idx in DATASET_INDICES:
    dataset_name, folder_url = choices[dataset_idx]
    print("\n" + "#"*80)
    print(f" STARTING DATASET: {dataset_name} ".center(80, "#"))
    print("#"*80)
    
    base = f"data/{dataset_name}"
    os.makedirs(base, exist_ok=True)
    rna_path = os.path.join(base, "adata_RNA.h5ad")
    
    if dataset_name.startswith("10x"):
        other_path = os.path.join(base, "adata_ADT.h5ad")
        annotation_path = os.path.join(base, "annotation.csv")
        gt_column = "manual-anno"
        data_type = '10x'
    else:
        other_path = os.path.join(base, "adata_ATAC.h5ad")
        annotation_path = os.path.join(base, "anno.csv")
        gt_column = "cluster"
        data_type = 'Spatial-epigenome-transcriptome'
        
    if not os.path.exists(rna_path) or not os.path.exists(other_path) or not os.path.exists(annotation_path):
        print(f"Downloading dataset files into: {base}")
        gdown_cmd = ".venv/bin/gdown" if os.path.exists(".venv/bin/gdown") else "gdown"
        os.system(f'{gdown_cmd} --folder "{folder_url}" --output "{base}"')
    else:
        print(f"Dataset files already exist at {base}. Skipping download.")
        
    adata_omics1 = sc.read_h5ad(rna_path)
    adata_omics2 = sc.read_h5ad(other_path)
    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()
    
    anno_df = pd.read_csv(annotation_path, index_col=0)
    adata_omics1.obs['ground_truth'] = anno_df[gt_column]
    adata_omics2.obs['ground_truth'] = anno_df[gt_column]
    
    sc.pp.filter_genes(adata_omics1, min_cells=10)
    if not dataset_name.startswith("10x"):
        sc.pp.filter_cells(adata_omics1, min_genes=200)
        
    sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_omics1, target_sum=1e4)
    sc.pp.log1p(adata_omics1)
    sc.pp.scale(adata_omics1)
    adata_omics1_high = adata_omics1[:, adata_omics1.var['highly_variable']]
    
    if dataset_name.startswith("10x"):
        adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=adata_omics2.n_vars - 1)
        adata_omics2 = clr_normalize_each_cell(adata_omics2)
        sc.pp.scale(adata_omics2)
        adata_omics2.obsm["feat"] = pca(adata_omics2, n_comps=adata_omics2.n_vars - 1)
    else:
        adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=50)
        adata_omics2 = adata_omics2[adata_omics1.obs_names].copy()
        if "X_lsi" not in adata_omics2.obsm:
            sc.pp.highly_variable_genes(adata_omics2, flavor="seurat_v3", n_top_genes=3000)
            lsi(adata_omics2, use_highly_variable=False, n_components=51)
        adata_omics2.obsm["feat"] = adata_omics2.obsm["X_lsi"].copy()
        
    data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)
    n_ground_truth = adata_omics1.obs["ground_truth"].nunique()
    print(f"Number of ground truth classes: {n_ground_truth}")
    
    dataset_results = []
    for seed in SEEDS:
        print(f"\n" + "-"*60)
        print(f" Dataset: {dataset_name} | Seed: {seed} ".center(60, "-"))
        print("-"*60)
        
        fix_seed(seed)
        data_copy = {'adata_omics1': data['adata_omics1'].copy(), 'adata_omics2': data['adata_omics2'].copy()}
        model = Train_scModuFuse(data_copy, datatype=data_type, device=device, 
                                 within_fusion=WITHIN_FUSION, between_fusion=BETWEEN_FUSION, 
                                 attention_type='local', random_seed=seed)
        output = model.train()
        
        adata = data_copy['adata_omics1'].copy()
        adata.obsm['scModuFuse'] = output['scModuFuse'].copy()
        clustering(adata, key='scModuFuse', add_key='scModuFuse', n_clusters=n_ground_truth, method=tool, use_pca=True, random_seed=seed)
        
        y_true = adata.obs['ground_truth'].astype(str)
        y_pred = adata.obs['scModuFuse'].astype(str)
        ari = adjusted_rand_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ami = adjusted_mutual_info_score(y_true, y_pred)
        homogeneity = homogeneity_score(y_true, y_pred)
        v_measure = v_measure_score(y_true, y_pred)
        
        joint_feat = adata.obsm['scModuFuse']
        le = LabelEncoder()
        y_pred_int = le.fit_transform(y_pred)
        sil_score = silhouette_score(joint_feat, y_pred_int)
        
        print(f"Result (Within: {WITHIN_FUSION}, Between: {BETWEEN_FUSION}) -> ARI: {ari:.4f} | NMI: {nmi:.4f} | Silhouette: {sil_score:.4f}")
        res_dict = {'dataset': dataset_name, 'seed': seed, 'ARI': ari, 'NMI': nmi, 'AMI': ami,
                    'Homogeneity': homogeneity, 'V-measure': v_measure, 'Silhouette': sil_score}
        dataset_results.append(res_dict)
        all_results.append(res_dict)
        
    df_ds = pd.DataFrame(dataset_results)
    df_ds.to_csv(f"results/scModuFuse_{WITHIN_FUSION}_{BETWEEN_FUSION}_{dataset_name}_results.csv", index=False)
    print("\n" + "="*80)
    print(f" SUMMARY FOR {dataset_name} ".center(80, "="))
    print("="*80)
    print(df_ds.describe().loc[['mean', 'std']])
    print("="*80)

df_all = pd.DataFrame(all_results)
df_all.to_csv(f"results/scModuFuse_{WITHIN_FUSION}_{BETWEEN_FUSION}_all_results.csv", index=False)

metrics_cols = ['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-measure', 'Silhouette']

print("\n" + "="*95)
print(f" FINAL BENCHMARK SUMMARY FOR scModuFuse (Within: {WITHIN_FUSION} / Between: {BETWEEN_FUSION}) ".center(95, "="))
print("="*95)

print("\n--- MEAN METRICS PER DATASET ---")
mean_table = df_all.groupby('dataset')[metrics_cols].mean()
print(mean_table.round(4).to_string())

print("\n--- MEAN ± STD METRICS PER DATASET ---")
summary_rows = []
for ds_name, group in df_all.groupby('dataset'):
    row = {'dataset': ds_name}
    for m in metrics_cols:
        row[m] = f"{group[m].mean():.4f} ± {group[m].std():.4f}"
    summary_rows.append(row)

df_summary_fmt = pd.DataFrame(summary_rows)
print(df_summary_fmt.to_string(index=False))

print("\n" + "-"*95)
overall_mean = df_all[metrics_cols].mean()
overall_std = df_all[metrics_cols].std()
overall_df = pd.DataFrame([
    overall_mean.round(4),
    overall_std.round(4),
    pd.Series([f"{m:.4f} ± {s:.4f}" for m, s in zip(overall_mean, overall_std)], index=metrics_cols)
], index=['Overall Mean', 'Overall Std', 'Overall Mean ± Std'])

print("\n" + "-"*95)
print(" OVERALL MEAN & STD ACROSS ALL DATASETS & SEEDS ".center(95, "-"))
print(overall_df.to_string())
print("="*95)
